# Marvedge Task-00046 — V4 Production Pipeline
**FFmpeg Pipe · YuNet CPU · Zero Seeks · FFmpeg-Native Crop**

No GPU required. Target: 22-min 11GB video in under 10 minutes.


In [ ]:
# CELL 1: System setup
import subprocess, os
os.system('apt-get install -qq ffmpeg 2>/dev/null')
os.system('pip install -q scenedetect[opencv] scipy tqdm opencv-python-headless')
# Verify YuNet is available via OpenCV
import cv2
assert hasattr(cv2, 'FaceDetectorYN'), 'Need opencv-python >= 4.5.4 for YuNet'
print(f'OpenCV {cv2.__version__} — YuNet available')
print('All dependencies ready')

In [ ]:
# CELL 2: Clone / pull repo (V4 pipeline is on this branch)
import os
BRANCH = 'feat/task-43-center-crop-fallback'
if not os.path.exists('/content/marvedge'):
    os.system(f'git clone -b {BRANCH} --depth 1 https://github.com/Marvedge/marvedge.git /content/marvedge')
else:
    os.system('git -C /content/marvedge pull')
os.chdir('/content/marvedge')
assert os.path.exists('scripts/ml/benchmark_preprocessing_v4.py'), 'V4 not found'
print('Repo ready — V4 pipeline confirmed')

In [ ]:
# CELL 3: Mount Google Drive + confirm video file
from google.colab import drive
import os, subprocess
drive.mount('/content/drive', force_remount=True)
print('Video files in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive/')):
    if any(f.lower().endswith(e) for e in ['.mp4','.mkv','.mov','.avi']):
        sz = os.path.getsize(f'/content/drive/MyDrive/{f}') / 1e9
        print(f'  {f}  ({sz:.2f} GB)')

In [ ]:
# CELL 4: Set video path  ← UPDATE FILENAME BELOW
import os, subprocess
VIDEO_FILENAME = 'kapil.mp4'   # <-- update to match what Cell 3 printed
VIDEO_PATH = f'/content/drive/MyDrive/{VIDEO_FILENAME}'
assert os.path.lexists(VIDEO_PATH), f'Not found: {VIDEO_PATH}'
dur = float(subprocess.check_output(
    ['ffprobe','-v','error','-show_entries','format=duration',
     '-of','default=noprint_wrappers=1:nokey=1', VIDEO_PATH]
).decode().strip())
sz = os.path.getsize(VIDEO_PATH) / 1e9
print(f'Video : {VIDEO_FILENAME}')
print(f'Size  : {sz:.2f} GB')
print(f'Duration: {dur/60:.1f} min ({dur:.0f}s)')

In [ ]:
# CELL 5: Run V4 pipeline
import time, json, os, subprocess
OUT_DIR = '/content/marvedge/demo/task46_v4'
REPORT  = f'{OUT_DIR}/benchmark_report.json'
print('=' * 65)
print('  V4 PRODUCTION PIPELINE')
print('  FFmpeg pipe + YuNet + FFmpeg-native crop')
print('=' * 65)
t0 = time.time()
result = subprocess.run(
    ['python', 'scripts/ml/benchmark_preprocessing_v4.py',
     '--videoPath',  VIDEO_PATH,
     '--savePath',   OUT_DIR,
     '--reportPath', REPORT,
     '--extractFps', '2',
     '--detScale',   '0.25',
     '--threads',    '2'],
    capture_output=False, text=True
)
t_total = time.time() - t0
if result.returncode != 0:
    print('Pipeline exited with error')
else:
    with open(REPORT) as f: rpt = json.load(f)
    print(f'DONE in {t_total:.1f}s ({t_total/60:.1f} min)')
    print(f'Throughput : {rpt["fps_throughput"]} fps')
    print(f'Realtime   : {rpt["realtime_ratio"]}x')
    print(f'Tracks     : {rpt["tracks_found"]} ({rpt["fallback_tracks_found"]} fallback)')

In [ ]:
# CELL 6: Download results JSON
import json
from google.colab import files
with open(REPORT) as f: rpt = json.load(f)
print('STAGE BREAKDOWN:')
for s in rpt['stages']:
    print(f"  {s['stage']:<30} {s['wall_time_sec']:>8.2f}s")
print(f"  {'TOTAL':<30} {rpt['total_wall_time_sec']:>8.2f}s")
out = '/content/task46_v4_results.json'
with open(out,'w') as f: json.dump(rpt, f, indent=2)
files.download(out)
print('Results downloaded')